In [ ]:
import numpy as np
from ufl.core.expr import Expr
from ufl import TestFunction, Measure
from lucifex.mesh import interval_mesh
from lucifex.fem import Constant, Function
from lucifex.fdm import ExplicitRungeKutta, RK4, DT, FunctionSeries, Series, ConstantSeries
from lucifex.solver import ibvp, evaluation , BoundaryConditions, rkibvp
from lucifex.sim import Simulation, run
from lucifex.viz import plot_line, save_figure
from lucifex.utils import nested_dict

from ufl import inner, grad


dt = 0.05
ics = ...
bcs = ...

rhs_auto = lambda v, u, a, d: -v * inner(a, grad(u)) + v * grad(d * grad(u))
rhs_nauto = lambda v, t, u, a, d: rhs_auto(v, u, a, d)


mesh = interval_mesh(1.0, 1)
u: FunctionSeries
t: FunctionSeries
a = ...
d = ...

u_auto_solver = rkibvp(rhs_auto, RK4, dt, ics, bcs)(u, a, d)
u_nauto_solver = rkibvp(rhs_nauto, RK4, dt, ics, bcs, autonomous=False)(t, u, a, d)

In [ ]:
def rk_advection_diffusion(
    u: FunctionSeries,
    t: ConstantSeries,
    dt: Constant | ConstantSeries,
    a: Series | Function | Expr,
    d: Series | Function | Expr,
    phi: Series | Function | Expr | float = 1,
    bcs: BoundaryConditions | None = None,
):
    """
    `∂u/∂t = - 𝐚·∇u + ∇·(D·∇u)`
    """
    v = TestFunction(u.function_space)
    dx = Measure('dx', u.function_space.mesh)
    a = Constant(u.function_space.mesh, a, name='a')
    d = Constant(u.function_space.mesh, d, name='d')
    F_dt = v * DT(u, dt) * dx
    
    F_rhs = RK4(rhs)(dt, t, u, a, d)
    return F_dt, -F_rhs